# Stock Focus Lab

Use this notebook for one-off analysis of explicitly chosen symbols without changing the shared daily or weekly shortlist notebooks.

- Edit `SYMBOLS` to the names you want to review
- Set `TIMEFRAME` to `"daily"` or `"weekly"`
- The notebook reads from `out/indicators/momentum_tv_match/<timeframe>/`
- Plot window defaults to the last 3 calendar years


In [ ]:
from pathlib import Path
import csv
import sys
from datetime import date, timedelta

import matplotlib.dates as mdates
import matplotlib.pyplot as plt

OUT_ROOT = Path("out")

SYMBOLS = ["NU", "RL", "GE", "NVDA", "C", "JPM", "BAC", "AAPL"]
TIMEFRAME = "weekly"
YEARS_TO_PLOT = 3
CHART_STYLE = "candles"  # "candles" or "line"

MACD_FAST = 12
MACD_SLOW = 26
MACD_SIGNAL = 9


def resolve_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "watchlists" / "watchlist.md").exists() and (candidate / OUT_ROOT).exists():
            return candidate
    return Path.cwd()


def read_rows(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def compute_ema(values: list[float], period: int) -> list[float | None]:
    out: list[float | None] = [None] * len(values)
    if period <= 0 or len(values) < period:
        return out
    alpha = 2.0 / (period + 1.0)
    seed = sum(values[:period]) / period
    out[period - 1] = seed
    prev = seed
    for idx in range(period, len(values)):
        current = (values[idx] - prev) * alpha + prev
        out[idx] = current
        prev = current
    return out


def compute_macd(
    values: list[float],
    fast: int,
    slow: int,
    signal: int,
) -> tuple[list[float | None], list[float | None], list[float | None]]:
    fast_ema = compute_ema(values, fast)
    slow_ema = compute_ema(values, slow)

    macd_line: list[float | None] = []
    for f_val, s_val in zip(fast_ema, slow_ema, strict=True):
        if f_val is None or s_val is None:
            macd_line.append(None)
        else:
            macd_line.append(f_val - s_val)

    valid_idx = [idx for idx, val in enumerate(macd_line) if val is not None]
    valid_vals = [float(macd_line[idx]) for idx in valid_idx]
    signal_compact = compute_ema(valid_vals, signal)

    signal_line: list[float | None] = [None] * len(macd_line)
    for compact_idx, full_idx in enumerate(valid_idx):
        signal_line[full_idx] = signal_compact[compact_idx]

    hist: list[float | None] = []
    for m_val, s_val in zip(macd_line, signal_line, strict=True):
        if m_val is None or s_val is None:
            hist.append(None)
        else:
            hist.append(m_val - s_val)

    return macd_line, signal_line, hist


def normalize_symbols(symbols: list[str]) -> list[str]:
    normalized: list[str] = []
    seen: set[str] = set()
    for raw in symbols:
        symbol = raw.strip().upper()
        if not symbol or symbol in seen:
            continue
        normalized.append(symbol)
        seen.add(symbol)
    return normalized


def timeframe_width_days(timeframe: str) -> float:
    return 1.9 if timeframe == "daily" else 6.0


def timeframe_recent_window(timeframe: str) -> int:
    return 5 if timeframe == "daily" else 2


def load_signal_engine_snapshot(path: Path) -> dict[str, dict[str, str]]:
    if not path.exists():
        return {}
    rows = read_rows(path)
    return {row.get("symbol", "").strip().upper(): row for row in rows if row.get("symbol")}


def last_event_date(rows: list[dict[str, str]], event_name: str) -> str:
    for row in reversed(rows):
        if (row.get("Event") or "").strip() == event_name:
            return (row.get("Date") or "").strip()
    return ""


def last_nonempty_event(rows: list[dict[str, str]]) -> str:
    for row in reversed(rows):
        event_name = (row.get("Event") or "").strip()
        if event_name:
            return event_name
    return ""


ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.plotting.candles import (
    build_ohlc_arrays,
    overlay_ema,
    overlay_event_markers,
    plot_candlesticks,
    plot_volume,
)

SYMBOLS = normalize_symbols(SYMBOLS)
if TIMEFRAME not in {"daily", "weekly"}:
    raise ValueError(f"Unsupported TIMEFRAME: {TIMEFRAME}")
if not SYMBOLS:
    raise RuntimeError("Populate SYMBOLS with at least one ticker.")

MOMENTUM_PATH = ROOT / OUT_ROOT / "indicators" / "momentum_tv_match" / TIMEFRAME
SIGNAL_ENGINE_LATEST = ROOT / OUT_ROOT / "_meta" / "latest" / "signal_engine_latest.csv"
RECENT_WINDOW_BARS = timeframe_recent_window(TIMEFRAME)
WIDTH_DAYS = timeframe_width_days(TIMEFRAME)
signal_snapshot = load_signal_engine_snapshot(SIGNAL_ENGINE_LATEST)

print(f"Repo root: {ROOT}")
print(f"Momentum dir: {MOMENTUM_PATH.relative_to(ROOT)}")
print(f"Timeframe: {TIMEFRAME}")
print(f"Recent-window bars for highlight: {RECENT_WINDOW_BARS}")
print("Symbols:", ", ".join(SYMBOLS))

rows_count = len(SYMBOLS)
fig, axes = plt.subplots(
    rows_count * 2,
    1,
    figsize=(13, 4.6 * rows_count),
    squeeze=False,
    gridspec_kw={"height_ratios": [3, 1] * rows_count},
)

for idx, symbol in enumerate(SYMBOLS):
    ax_price = axes[idx * 2][0]
    ax_macd = axes[idx * 2 + 1][0]
    csv_path = MOMENTUM_PATH / f"{symbol}.csv"

    if not csv_path.exists():
        ax_price.set_title(f"{symbol} ({TIMEFRAME}) | missing data file")
        ax_price.axis("off")
        ax_macd.axis("off")
        continue

    rows = read_rows(csv_path)
    ohlc = build_ohlc_arrays(rows)
    dates = ohlc["dates"]
    opens = ohlc["opens"]
    highs = ohlc["highs"]
    lows = ohlc["lows"]
    closes = ohlc["closes"]
    volumes = ohlc["volumes"]

    events_by_date: dict[date, str] = {}
    for row in rows:
        raw_day = (row.get("Date") or "").strip()
        if not raw_day:
            continue
        try:
            day = date.fromisoformat(raw_day)
        except ValueError:
            continue
        events_by_date[day] = (row.get("Event") or "").strip()

    if not dates:
        ax_price.set_title(f"{symbol} ({TIMEFRAME}) | no rows")
        ax_price.axis("off")
        ax_macd.axis("off")
        continue

    ema50 = compute_ema(closes, 50)
    ema200 = compute_ema(closes, 200)
    events = [events_by_date.get(day, "") for day in dates]
    macd_line, macd_signal, macd_hist = compute_macd(closes, MACD_FAST, MACD_SLOW, MACD_SIGNAL)

    start_date = dates[-1] - timedelta(days=365 * YEARS_TO_PLOT)
    start_idx = next((i for i, day in enumerate(dates) if day >= start_date), 0)

    plot_dates = dates[start_idx:]
    plot_opens = opens[start_idx:]
    plot_highs = highs[start_idx:]
    plot_lows = lows[start_idx:]
    plot_closes = closes[start_idx:]
    plot_volumes = volumes[start_idx:]
    plot_ema50 = ema50[start_idx:]
    plot_ema200 = ema200[start_idx:]
    plot_events = events[start_idx:]
    plot_macd = macd_line[start_idx:]
    plot_signal = macd_signal[start_idx:]
    plot_hist = macd_hist[start_idx:]

    if CHART_STYLE == "candles":
        plot_candlesticks(
            ax_price,
            plot_dates,
            plot_opens,
            plot_highs,
            plot_lows,
            plot_closes,
            width_days=WIDTH_DAYS,
        )
    elif CHART_STYLE == "line":
        ax_price.plot(plot_dates, plot_closes, label="Close", color="#1f77b4", linewidth=1.2)
    else:
        raise ValueError(f"Unsupported CHART_STYLE: {CHART_STYLE}")

    overlay_ema(ax_price, plot_dates, plot_ema50, label="EMA 50", color="#ff7f0e", linewidth=1.0)
    overlay_ema(ax_price, plot_dates, plot_ema200, label="EMA 200", color="#2ca02c", linewidth=1.0)

    overlay_event_markers(
        ax_price,
        plot_dates,
        plot_closes,
        plot_events,
        marker_map={
            "MomLE": {
                "label": "MomLE",
                "marker": "^",
                "size": 26,
                "color": "#0a84ff",
                "alpha": 0.85,
                "anchor": "low",
                "y_offset_frac": -0.02,
            },
            "MomSE": {
                "label": "MomSE",
                "marker": "v",
                "size": 24,
                "color": "#d81b60",
                "alpha": 0.55,
                "anchor": "high",
                "y_offset_frac": 0.02,
            },
        },
    )

    recent_slice_start = max(0, len(rows) - RECENT_WINDOW_BARS)
    recent_rows = rows[recent_slice_start:]
    recent_buy_dates = {
        date.fromisoformat(row["Date"])
        for row in recent_rows
        if (row.get("Event") or "").strip() == "MomLE"
    }
    recent_momle_x = [
        day
        for day, event_name in zip(plot_dates, plot_events, strict=True)
        if event_name == "MomLE" and day in recent_buy_dates
    ]
    recent_momle_y = [
        price
        for day, price, event_name in zip(plot_dates, plot_closes, plot_events, strict=True)
        if event_name == "MomLE" and day in recent_buy_dates
    ]

    if recent_momle_x:
        ax_price.scatter(
            recent_momle_x,
            recent_momle_y,
            marker="^",
            s=68,
            color="#00c853",
            edgecolors="black",
            linewidths=0.7,
            label=f"Recent MomLE ({RECENT_WINDOW_BARS} bars)",
            zorder=8,
        )

    ax_volume_overlay = ax_price.twinx()
    plot_volume(
        ax_volume_overlay,
        plot_dates,
        plot_opens,
        plot_closes,
        plot_volumes,
        width_days=WIDTH_DAYS,
        alpha=0.18,
    )
    ax_volume_overlay.set_yticks([])
    ax_volume_overlay.set_ylabel("")
    ax_volume_overlay.grid(False)

    x_num = mdates.date2num(plot_dates)
    for x_val, hist_value in zip(x_num, plot_hist, strict=True):
        if hist_value is None:
            continue
        color = "#2ca02c" if hist_value >= 0 else "#d62728"
        ax_macd.bar(x_val, hist_value, color=color, alpha=0.35, width=WIDTH_DAYS)

    ax_macd.plot(plot_dates, plot_macd, label="MACD", color="#1f77b4", linewidth=1.3)
    ax_macd.plot(plot_dates, plot_signal, label="Signal", color="#ff7f0e", linewidth=1.3)
    ax_macd.axhline(0.0, color="#111111", linewidth=1.0, alpha=0.6, linestyle="--")

    latest_snapshot = signal_snapshot.get(symbol, {})
    trend_status = (latest_snapshot.get("Trend") or "-").strip() or "-"
    momentum_state = (latest_snapshot.get("MomentumState") or "-").strip() or "-"
    last_momle = last_event_date(rows, "MomLE") or "-"
    last_event = last_nonempty_event(rows) or "-"
    latest_close = plot_closes[-1]

    ax_price.set_title(
        f"{symbol} ({TIMEFRAME}) | close: {latest_close:.2f} | trend: {trend_status} | "
        f"momentum: {momentum_state} | last event: {last_event} | last MomLE: {last_momle}"
    )
    ax_price.grid(alpha=0.25)
    ax_price.legend(loc="best", fontsize=8)
    ax_price.tick_params(axis="x", labelbottom=False)

    ax_macd.set_ylabel("MACD", fontsize=8)
    ax_macd.grid(alpha=0.2)
    ax_macd.legend(loc="best", fontsize=8)
    ax_macd.xaxis.set_major_locator(mdates.YearLocator())
    ax_macd.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.suptitle(
    f"Stock Focus Lab: {TIMEFRAME.title()} view for {', '.join(SYMBOLS)}",
    fontsize=14,
)
fig.tight_layout(rect=[0, 0, 1, 0.97], h_pad=0.25)
plt.show()

